In [ ]:
# Install everything you need for this homework
%pip install gitsource minsearch openai toyaikit

In [3]:
import gitsource
# get package version if available
gitsource_version = getattr(gitsource, "__version__", None)
gitsource_version


'0.0.5'

In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
# Parse each file into a dictionary
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

# Q1 Answer:
print("Number of lesson pages:", len(documents))

Number of lesson pages: 72


### 🔍 Q2 — Index and Search

In [ ]:
from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [7]:
question = "How does the agentic loop keep calling the model until it stops?"
# Search with the given query
def search(question):
    return index.search(question)


search_results = search(question)

# Q2 Answer:
print("Top result filename:", search_results[0]["filename"])

Top result filename: 01-agentic-rag/lessons/14-agentic-loop.md


### 💬 Q3 — Basic RAG (Read + Answer)

In [10]:
def build_context(search_results):
    lines = []
    for doc in search_results:
           lines.append("File:" + doc['filename'])
           lines.append("Content:" + doc['content'])
    return "\n".join(lines).strip()

In [11]:
# Safely build context only when search_results exists and is non-empty
if 'search_results' in globals() and search_results:
    context = build_context(search_results)
else:
    context = ""
    print("No search_results found — run the Q2 cells (index/search) to generate them.")

In [ ]:
print(context)

In [14]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = f"""
    You are a helpful assistant that answers questions based on the provided context.
    
    Context:
    {context}
    
    Question:
    {question}
    
    Answer:
    """
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)
print(prompt)

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GROQ_API_KEY")

In [17]:
from openai import OpenAI
import os
# Resolve API key from environment if available (OPENAI_API_KEY preferred, fallback to GROQ_API_KEY)
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY or GROQ_API_KEY environment variable before creating OpenAI client.")
openai_client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [19]:
response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=[
        {"role": "system", "content": "You are a helpful assistant that answers questions based on the provided context."},
        {"role": "user", "content": prompt}
    ],
    #max_output_tokens=500,
    temperature=0.7,
)

In [20]:
print(response.output_text)

The agentic loop keeps calling the model until it stops by using a `while` loop that continues to run as long as the model returns a response with a function call. The loop sends the model's output back to the model as input, along with the previous conversation history, until the model returns a response without a function call. This allows the model to decide when to stop and return a final answer.


In [21]:
instructions = "You are a helpful assistant that answers questions based on the provided context."

### The LLM function

In [ ]:
import json

In [22]:
def llm(instructions, user_prompt, model="llama-3.3-70b-versatile"):
    """Call the Responses API with a normalized, serializable message history."""
    # Build a clean list of message dicts (ensure strings)
    message_history = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    
    answer = response.output_text
    input_tokens = response.usage.input_tokens

    return answer, input_tokens

In [23]:
answer = llm(instructions, prompt, model="llama-3.3-70b-versatile")

In [25]:
def rag(query, model="llama-3.3-70b-versatile"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(instructions, prompt, model=model)
    return answer

In [26]:
# Question 3 Answer:
answer, tokens = rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(tokens)

The agentic loop keeps calling the model until it stops by using a `while True` loop that continues to send messages to the model and execute any function calls until the model returns a response without any function calls. This is achieved through a flag `has_function_calls` that is set to `True` if the model's response contains a function call, and `False` otherwise. The loop breaks when `has_function_calls` is `False`, indicating that the model has finished its task and does not need to make any further function calls.
10431


### ✂️ Q4 — Chunking (Cutting Pages into Sections)

In [27]:
from gitsource import chunk_documents

In [28]:
chunks = chunk_documents(documents, size=2000, step=1000)

# Q4 Answer:
print("Number of chunks:", len(chunks))

Number of chunks: 295


### 🔍💬 Q5 — RAG with Chunking

In [33]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

In [34]:
def search_chunked(query):
    return chunk_index.search(query)

In [35]:
def rag_chunked(query, model="llama-3.3-70b-versatile"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(instructions, prompt, model=model)
    return answer

In [36]:
answer2, tokens2 = rag_chunked("How does the agentic loop keep calling the model until it stops?")
print("Chunked input tokens:", tokens2)
print("Reduction factor:", round(tokens / tokens2, 1), "x fewer")  # Q5 Answer

Chunked input tokens: 10431
Reduction factor: 1.0 x fewer


### 🤖 Q6 — Turn it Into an Agent

In [95]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [96]:
def search(query: str) -> list:
    """Search the course lessons for information about the given query."""
    return chunk_index.search(query)

In [97]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [98]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons for information about the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [102]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [105]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="llama-3.1-8b-instant", client=openai_client)
)

In [106]:
result = runner.loop("How does the agentic loop work, and how is it different from plain RAG?", callback=callback,)
print(result)

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseReasoningItem(id='resp_01kvqr80xefbrvkmd8q3ebvk3c', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop vs plain RAG"}', call_id='p6x41ct85', name='search', type='function_call', id='p6x41ct85', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'p6x41ct85', 'output': '[\n  {\n    "start": 4000,\n    "content": "esult.\\n\\nThe `result` is a `LoopResult` with `all_messages` (the full\\nconversation), token counts, and `cost` (computed from token usage).\\n\\n## Cost and token

/usr/local/lib/python3.12/dist-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'llama-3.1-8b-instant'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
